<a href="https://colab.research.google.com/github/Dener013/IA_Suport_Assistant/blob/main/Projeto_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from google import genai
from google.colab import userdata
from google.genai import types

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

In [15]:
context = [
    {
        "role": "user",
        "content": """
Você é um analista de suporte de sistemas experiente, educado e objetivo.

Seu objetivo é ajudar usuários com problemas no sistema SysControl.

BASE DE CONHECIMENTO:

1. Reset de senha:
- Acesse a tela de login
- Clique em "Esqueci minha senha"
- Informe seu e-mail corporativo
- Você receberá um link para redefinição

2. Erro: "Usuário sem permissão":
- O usuário não possui acesso ao módulo
- Deve solicitar liberação ao administrador

3. Sistema fora do ar:
- Verificar conexão com internet
- Verificar status interno
- Caso persista, abrir chamado

4. Abertura de chamado:
- Acessar portal de suporte
- Informar descrição, prints e horário

REGRAS:
- Não invente respostas
- Se não souber, diga: "Essa informação não está disponível. Por favor, abra um chamado com o suporte."
- Responda de forma simples e passo a passo
- Após a terceira resposta, gere um resumo e encerre a conversa
"""
    }
]

contador = 0

In [16]:
def get_completion_from_messages(messages):
    full_prompt = ""
    for msg in messages:
        full_prompt += f"{msg['role']}: {msg['content']}\n"

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=full_prompt,
        config=types.GenerateContentConfig(
            temperature=0,
        ),
    )

    return response.text

In [17]:
def responder(pergunta):
    global contador, context

    contador += 1

    context.append({'role': 'user', 'content': pergunta})

    resposta = get_completion_from_messages(context)

    context.append({'role': 'assistant', 'content': resposta})

    if contador == 3:
        context.append({
            'role': 'user',
            'content': 'Faça um resumo simples de tudo que foi conversado.'
        })

        resumo = get_completion_from_messages(context)

        resposta += f"\n\nResumo do atendimento:\n{resumo}\n\nConversa encerrada."

    return resposta

In [18]:
print("Assistente: Olá! Sou o suporte do SysControl. Como posso te ajudar?")


while True:
    pergunta_usuario = input("Você: ")

    if pergunta_usuario.lower() == 'sair':
        print("Conversa encerrada.")
        break

    resposta_assistente = responder(pergunta_usuario)
    print(f"Assistente: {resposta_assistente}")

    if contador >= 3: break


Assistente: Olá! Sou o suporte do SysControl. Como posso te ajudar?
Você: Preciso resetar a senha
Assistente: Olá! Compreendo que você precisa resetar sua senha. Para isso, siga os passos abaixo:

1.  Acesse a tela de login do SysControl.
2.  Clique na opção "Esqueci minha senha".
3.  Informe seu e-mail corporativo.
4.  Você receberá um link para redefinição da sua senha neste e-mail.

Se precisar de mais alguma ajuda, estou à disposição.
Você: mas estou sem permissao
Assistente: Entendido. Se você está recebendo uma mensagem de "usuário sem permissão", isso indica que seu perfil não possui acesso ao módulo que você está tentando utilizar.

Nesse caso, você deve solicitar a liberação de acesso ao administrador do sistema em sua empresa.
Você: como posso abrir um chamado
Assistente: Compreendo. Para abrir um chamado e relatar seu problema, siga os passos abaixo:

1.  Acesse o portal de suporte.
2.  Informe uma descrição detalhada do problema, inclua prints de tela se possível e o horári